In [6]:
#  Установка зависимостей
!pip install -q -v transformers accelerate datasets

In [7]:
#импорт библиотек
import os
import json
import random
import copy
import numpy as np # linear algebra
import pandas as pd # data processing
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import urllib.request
from datasets import load_dataset

In [8]:
#Настройка среды, загрузка llm модели

#фиксация воспроизводимости рандомных операций
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)#seed для генератора случайных чисел pytorch на сpu
torch.cuda.manual_seed_all(SEED) #seed для всех генераторов случайных чисел pytorch на всех доступных gpu

#назначение вычислительного устройства
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" 
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUT_DIR = "artifacts"
os.makedirs(OUT_DIR, exist_ok=True) #по умолчанию создаст out_dir в kaggle/working

print(f"Работаем на {DEVICE}, модель: {MODEL_NAME}")

# Загрузка токенизатора. 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#Загрузка модели. 
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16, 
    device_map=DEVICE, #указываем, на какое устройство поместить модель
)
model.eval() 
print("Модель загружена")

# Сохраняем оригинальную модель для будущих сравнений
model_original = model  

N_LAYERS = model.config.num_hidden_layers #количество слоёв у модели
D_MODEL = model.config.hidden_size 
print(f"Слоёв: {N_LAYERS}, размерность: {D_MODEL}")

#конвертация индекса hidden state -> decoder layer
def hs_index_to_decoder_layer(hs_idx: int) -> int:
    return max(0, hs_idx - 1) #Индекс слоя на 1 меньше, чем индекс его выхода

# Настройка токенизатора для корректной работы с батчами
tokenizer.padding_side = "left" 
if tokenizer.pad_token is None: #проверяем, есть ли у токенизатора падинговый токен (без него падинг будет падать с ошибкой)
    tokenizer.pad_token = tokenizer.eos_token #если нет, присваиваем ему токен конца последовательности

Работаем на cuda, модель: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Модель загружена
Слоёв: 28, размерность: 1536


Устанавливаем фиксированное зерно во все места, где будет использоваться случайность для воспроизводимости

Назначаем главное вычислительное устройство: если в системе есть доступная видеокарта NVIDIA с установленным CUDA и PyTorch с поддержкой CUDA, то ставим её, если нет, то cpu

# Загрузка модели и токенизатора:
* Токенизатор преобразует текст в числовые токены, которые потом эмбединги преобразят в вектора
* AutoTokenizer автоматически определяет правильный класс токенизатора для указанной модели
* AutoModelForCausalLM загружает предобученную модель c HuggingFace
* Параметр torch_dtype загружает веса в формате половинной точности (16 бит на каждый вес вместо 32) для экономии оперативной и видеопамяти
* С помощью model.eval() переключаем модель в режим оценки (для инференса работает немного иначе), потому что мы не будем её обучать, а лишь делать forward-pass проходы
* Сохраняем количество слоёв модели и размерность каждого токена после превращения в вектор (тензор) эмбедингами и попадания в остаточный поток

Напишем вспомогательную функцию для превращения индекса hidden_states в индекс decoder layer (всегда меньше на 1)
hidden_states - это выход слоя (residual stream)

# Настройка токенизатора:
* Т к у модели не бесконечное количество памяти, мы разбиваем пользовательский промпт на батчи с последовательностями токенов [batch_size, seq_len, d_model], токены в батче обрабатываются слоём параллельно, однако батчи обрабатываются слоями последовательно: сколько батчей, столько и forward passes

* Устанавливаем, с какой стороны токенизатор будет добавлять паддинг (специальные токены,чтобы все последовательности в батче были одинаковой длины). 
* По умолчанию в большинстве токенизаторов стоит "right", а нам нужно "left", чтобы реальные данные были прижаты к правому краю, откуда мы и берём последний токен residual stream

* При стандартной генерации батчей и последовательностей нет, модель кушает один промпт (последовательность) и генерирует для него ответ (на который отказ или нет), в нашем режиме для убыстрения генерации активаций, мы передаём в модель те же отдельные промпты, но целыми батчами, чтобы обрабатывать каждый батч параллельно, используя модель на максимальную мощность

In [9]:
# Загрузка датасетов

# AdvBench (вредные промпты)
ADV_URL = "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv"
adv_path = os.path.join(OUT_DIR, "harmful_behaviors.csv")
if not os.path.exists(adv_path):
    urllib.request.urlretrieve(ADV_URL, adv_path) 
adv_df = pd.read_csv(adv_path)
harmful_prompts_all = adv_df["goal"].tolist() 
print(f"Загружено вредных промптов: {len(harmful_prompts_all)}")

# Alpaca (безвредные промпты) - берём только инструкции без input
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
alpaca_simple = [x["instruction"] for x in alpaca if x["input"] == ""]
random.shuffle(alpaca_simple)
harmless_prompts_all = alpaca_simple[: len(harmful_prompts_all)]
print(f"Загружено безвредных промптов: {len(harmless_prompts_all)}")

# Загрузка Wikitext для будущих метрик деградации
wikitext = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
wikitext_sentences = [t for t in wikitext["text"] if len(t.strip()) > 20]
# Сохраним в artifacts
with open(os.path.join(OUT_DIR, "wikitext_sentences.json"), "w") as f:
    json.dump(wikitext_sentences[:300], f) 
    
#Разделение на Train / Val / Held-out 
random.shuffle(harmful_prompts_all)
random.shuffle(harmless_prompts_all)

N_TRAIN = 300
N_VAL = 70
N_HELDOUT = 150

harmful_train = harmful_prompts_all[:N_TRAIN]
harmful_val = harmful_prompts_all[N_TRAIN:N_TRAIN + N_VAL]
harmful_heldout = harmful_prompts_all[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_HELDOUT]

harmless_train = harmless_prompts_all[:N_TRAIN]
harmless_val = harmless_prompts_all[N_TRAIN:N_TRAIN + N_VAL]
harmless_heldout = harmless_prompts_all[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_HELDOUT]

#Для проверки стабильности направления делим harmful train на две половины
harmful_train_sh = harmful_train.copy()
random.shuffle(harmful_train_sh)
half_A = harmful_train_sh[: N_TRAIN // 2]
half_B = harmful_train_sh[N_TRAIN // 2:]

print(f"Train: {len(harmful_train)} вредных / {len(harmless_train)} безвредных")
print(f"Val: {len(harmful_val)} / {len(harmless_val)}")
print(f"Held-out: {len(harmful_heldout)} / {len(harmless_heldout)}")
print(f"Half_A: {len(half_A)}, Half_B: {len(half_B)}")

# Сохраняем списки промптов для будущих частей (B, C, D)
with open(os.path.join(OUT_DIR, "data_splits.json"), "w", encoding="utf-8") as f:
    json.dump({
        "harmful_train": harmful_train,
        "harmless_train": harmless_train,
        "harmful_val": harmful_val,
        "harmless_val": harmless_val,
        "harmful_heldout": harmful_heldout,
        "harmless_heldout": harmless_heldout,
        "half_A": half_A,
        "half_B": half_B,
    }, f, ensure_ascii=False, indent=2)

#Функция форматирования промптов через chat template
def format_prompt(instruction: str) -> str:
    messages = [{"role": "user", "content": instruction}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

Загружено вредных промптов: 520


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Загружено безвредных промптов: 520


...

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train: 300 вредных / 300 безвредных
Val: 70 / 70
Held-out: 150 / 150
Half_A: 150, Half_B: 150


# Загрузка датасетов:

* С помощью urllib устанавливаем сырой файл данных с вредными запросам из github репозитория, если он ещё не установлен

* В adv датасете вредных промптов берём только goal игнорируем target (эталонный ответ на вредный запрос), превращаем столбец датафрейма в список промптов

* Полезные инструкции из альпака загружаем с помощью load_dataset (которая загружает датасеты с hugging face, а датасеты, загруженные с хаггинг фейс хранятся в специальной папке кэша кагл, поэтому мы не кладём их в артефакты), беря сплит трэйна, потому что других нет в данном датасете на 52 тыс инструкций. В список наших датасетов включаются только инструкции и лишь те инстуркции, у которых нет контекста input, тобеж он являются самодостаточными (всего в alpaca три колонки input, instruction и output), перемешиваем датасет, чтобы взять случайные первые 520 инструкций.


* Берём как промпты (300 штук) из вики текст (там одна колонка) только те предложения, в которых больше 20 символов без пробельных. Затем с помощью json.dump пишем и сохраняем пайтон-объект (список в данному случае) в указанный файл в формате json.

# Разделение данных на выборки:

* Делим обе выборки (полезных и вредных промптов) на тренировочную 300 (для нахождения different-in-means по слоям и расчёта Heatmap), валидационную 70 (для выбора оптимального слоя, практика взята на вооружение из статьи), отложенную 150 (для метрик деградации и доли/вероятности отказов модели).

* Также делим тренировочную выборку вредных промптов на две части (перед этим перемешивая harmful_train, на котором они базируются для случайного разделения), каждая из которых в расчёте косинусной близости для heatmap скомпонуется с harmless_train по среднему.


* Сохраняем списки промптов в json файл с помощью json.dump и помещаем его в kaggle/working/artifacts, чтобы можно было в следующих ячейках обращаться к выборкам (даже с началом новой сессии), не перезапуская все ячейки

* Пишем специальную функцию, которая превращает сырой промпт во входные данные, к которым модель привыкла, когда обучалась, с именем пользователя, инструкцией, токенами разделения и токеном конца запроса пользователя, подразумевающего, что ей надо формировать ответ в формате ассистента
смысл в том, чтобы модель обучена реагировать не только на входной набор слов из инструкции, но и на специальные шаблонные  токены, которые говорят об имени юзера его инструкции, и в частности токен добавляемый параметром add_generation_prompt=True обозначает конец запроса пользователя и говорит модели, что ей нужно именно отвечать на запрос как асситент, а не что то другое (именно этот токен промпта мы и будем брать для вычисления  refusal direction)

In [10]:
# функция для сбора активаций с residual stream (последний токен каждого промпта с каждого из слоёв)
@torch.no_grad()
def get_residual_activations(prompts, batch_size=8):
    """
    Возвращает тензор [N_LAYERS+1, n_prompts, d_model] - целевые активации
    на последнем токене для каждого слоя для каждого промпта.
    """
    all_acts = [[] for _ in range(N_LAYERS + 1)]
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size] #берём срез из списка промптов длинной батча
        formatted = [format_prompt(p) for p in batch] #форматируем под чат темплейт все промпты в батче
        enc = tokenizer(
            formatted, return_tensors="pt", padding=True, truncation=True
        ).to(DEVICE)

        out = model(**enc, output_hidden_states=True) #ставим параметр вывода остаточного потока активаций
        hs = out.hidden_states  # извлекаем остаточные потоки

        #проверяем коррректность падинга
        assert tokenizer.padding_side == "left"

        for layer_idx, h in enumerate(hs): #пронумеровываем все выходы слоёв
            last_tok = h[:, -1, :]  
            all_acts[layer_idx].append(last_tok.float().cpu())

    return torch.stack([torch.cat(layer_list, dim=0) for layer_list in all_acts])

#сбор активаций
print("Собираем активации: harmful_train ...")
acts_harmful_train = get_residual_activations(harmful_train)

print("Собираем активации: harmless_train ...")
acts_harmless_train = get_residual_activations(harmless_train)

print("Собираем активации: half_A ...")
acts_half_A = get_residual_activations(half_A)

print("Собираем активации: half_B ...")
acts_half_B = get_residual_activations(half_B)

# Сохраняем активации на диск (чтобы не пересчитывать)
torch.save({
    "acts_harmful_train": acts_harmful_train,
    "acts_harmless_train": acts_harmless_train,
    "acts_half_A": acts_half_A,
    "acts_half_B": acts_half_B,
}, os.path.join(OUT_DIR, "activations.pt"))

print("Активации сохранены в", os.path.join(OUT_DIR, "activations.pt"))

#Сохраняем метаданные для воспроизводимости
metadata = {
    "seed": SEED,
    "model_name": MODEL_NAME,
    "n_layers": N_LAYERS,
    "d_model": D_MODEL,
    "n_train": N_TRAIN,
    "n_val": N_VAL,
    "n_heldout": N_HELDOUT,
}
with open(os.path.join(OUT_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Метаданные сохранены")
print("=" * 60)
print("Артефакты в папке:", OUT_DIR)
print("Файлы:")
print("  - artifacts/activations.pt")
print("  - artifacts/data_splits.json")
print("  - artifacts/metadata.json")
print("  - artifacts/harmful_behaviors.csv")
print("  - artifacts/wikitext_sentences.json") #нейтральный корпус

Собираем активации: harmful_train ...
Собираем активации: harmless_train ...
Собираем активации: half_A ...
Собираем активации: half_B ...
Активации сохранены в artifacts/activations.pt
Метаданные сохранены
Артефакты в папке: artifacts
Файлы:
  - artifacts/activations.pt
  - artifacts/data_splits.json
  - artifacts/metadata.json
  - artifacts/harmful_behaviors.csv
  - artifacts/wikitext_sentences.json


# Сбор и сохранение активаций:
* Перед написанием функции для сбора активаций пишем декоратор, отключающий автоматическое запоминание градиентов для обратного прохода, т к у нас только форвард пасс проходы, нам не нужно обучать модель, следовательно можно не тратить ресурсы на запоминание градиентов.

* Размер батча для сбора активаций на промптах для нашей модели выбираем 8 с расчётом на то, что мы используем T4, это компромисс между скоростью сбора активаций и надёжностью (чтобы не переполнять видеопамять). Ограничение размера батча идёт не от самой архитектуры модели, а от видеопамяти на gpu.

* Создаём список для будущих hidden states длинной 25. Создаём цикл, внутри которого проходимся по нашим промптам с шагом размера батча. Берём срез из промптов длинной батча, форматируем внутри него все промпты чтобы они были читаемыми нашей моделью

* Токенизируем отформатированные промпты с помощью нашего токенайзера. Параметр return_tensors="pt" возвращает тензоры PyTorch. padding=True добавляет паддинг до максимальной длины в батче (использует padding_side="left", который мы установили ранее), параметр truncation=True обрезает слишком длинные последовательности (до лимита токенизатора 32 тыс токенов у нашего), to(DEVICE) — переносит тензоры на GPU (или CPU), чтобы данные и модель были на одном устройстве, это требуется для того, чтобы можно было корректно передать одно в другое.

* В Результате на выходе токенайзера у нас словарь с полями: input_ids  тензор [batch_size, max_len] с токенами,
attention_mask тензор [batch_size, max_len] с 1 для реальных токенов и 0 для паддинга,
который мы потом распаковываем как именованные аргументы, передавая на вход модели, указывая, что на выходе нужно также выдавать остаточный поток активаций каждого слоя. И выполняем.

* С выхода модели извлекаем все остаточные потоки всех промптов со всех слоёв. Проверяем корректность падинга токенайзера.

* Из всех активаций промптов (количество промптов ограничено текущим батчом) текущего слоя оставляем активации только последнего токена и складываем активации этих последний токенов по слоям в наш список, который сформировали вначале по 8 штук (батч) за итерацию цикла, добавляя с помощью аппенд в all_acts[layer_idx] список из промптов длинной размерности батча (перед этим переводим вектор в float32, т к у нас в моделе float16 и .cpu() переносим вектор на cpu, чтобы освободить видеопамять), чтобы активации батчей шли последовательно в правильном порядке.
Берём именно последний токен, потому что на этой позиции модель уже обработала весь запрос и готова начать генерацию. Именно здесь формируется решение об отказе.

У нас весь код выполняется не на gpu, а на gpu выполняется только часть с вычислениями с нейронкой, поэтому перед тем как подавать данные в нейронку, мы переносим их на gpu, а после выхода данных из нейронки, мы сразу переносим обратно на cpu, чтобы не засорять видеопамять и чтобы быстрее происходили вычисления.

* Теперь с помощью torch.cat склеиваем батчи внутри каждого слоя из all_acts по первому измерению dim=0 (т е по количеству промптов) и в каждом слое получается один большой тензор, где каждый элемент - активация последнего токена определённого промпта. А затем с помощью torch.stack все тензоры слоёв получившиеся объединяем в один тензор, рождая новое измерение N_LAYERS+1 (т к ещё hidden_state со слоя эмбеддинга)

* Собираем 4 набора активаций: для генерации и отсева кандидатов на refusal direction (вредные, безвредные) и для heatmap две половины вредных (лечге пересчитать ещё раз, т к у нас эти наборы перемешаны по сравнению с оригинальным harmful_train)

* С помощью torch.save сохраняем эти наборы активаций как Kaggle/working/artefacts/activations.pt

* На всякий случай сохраняем метаданные о модели и выборках  с помощью json.dump


In [11]:
#проверка корректностей размерностей сохранённых активаций
data = torch.load("artifacts/activations.pt")
print(data["acts_harmful_train"].shape)   
print(data["acts_harmless_train"].shape)  
print(data["acts_half_A"].shape)          
print(data["acts_half_B"].shape)

torch.Size([29, 300, 1536])
torch.Size([29, 300, 1536])
torch.Size([29, 150, 1536])
torch.Size([29, 150, 1536])
